## Eksāmena darbā izmantotais kods

Pirmkārt, tiek instalētas nepieciešamās bibliotēkas. Darbam tiek izmantots Beautiful Soup datu izgūšanai no failiem un Stanza teksta vārdu semantisko lomu marķēšanai.

In [ ]:
%pip install beautifulsoup4
%pip install stanza

Kad nepieciešami rīki ir instalēti, sāk ar datu izgūšanu no failiem, to satīrīšanu un salikšanu vienotā struktūrā.

In [93]:
from bs4 import BeautifulSoup
import os
import re

# Papildfunkcija, kas skaitli pārvērš par divu simbolu garu virkni.
def twoString(x):
    if (x < 10):
        return "0" + str(x)
    else:
        return str(x)
# Papildfunkcija, kas skaitli pārvērš par trīs simbolu garu virkni.
def threeString(x):
    if (x < 10):
        return "00" + str(x)
    elif (x < 100):
        return "0" + str(x)
    else:
        return str(x)

directories = [x[0] for x in os.walk("text")] # Atrod visus ceļus uz dažādajām.

# Iziet cauri visiem sējumiem un no failiem izvelk tekstus, apvienojot tos vienotā struktūrā pēc attīrīšanas.
all_text = []
for directory in directories:
    if directory.count('\\') > 1:   # Tiek izņemtas visas mapes, kurās neglabājas faili.
        files = []
        for (dirpath, dirnames, filenames) in os.walk(directory):
            files.extend(filenames)
            break

        file_num = 1
        for file in files:    
            id = directory + '\\' + threeString(file_num)

            with open(directory+'\\'+file, encoding="windows-1257") as page:
                soup = BeautifulSoup(page.read(), "html.parser")
                extracted_paragraphs = soup.find_all('p')
                extracted_text = ""
                p_count = 0
                y_flag = False
                for p in extracted_paragraphs:
                    if p_count > 1:
                        temp_text = p.get_text()
                        if temp_text == "":
                            continue
                        if temp_text[0].isdigit():
                            continue
                        if "y" in temp_text:
                            y_flag = True
                            break
                        
                        # Teksta tīrīšana.
                        temp_text = re.sub('[ ]+', ' ', temp_text)      # Astarpju un jaunu rindiņu apstrāde.
                        temp_text = temp_text.replace('\n', ' ')

                        temp_text = temp_text.replace('`', '\'')        # Pēdiņu apstrāde.
                        temp_text = temp_text.replace('\'\'', '"')
                        temp_text = temp_text.replace('\'', '')

                        temp_text = temp_text.replace('„', '"')
                        temp_text = temp_text.replace('«', '"')
                        temp_text = temp_text.replace('»', '"')

                        temp_text = re.sub(r'\d+', "100", temp_text)    # Skaitļu apstrāde.

                        temp_text = temp_text.replace('[', '')          # Iekavu apstrāde.
                        temp_text = temp_text.replace(']', '')
                        
                        temp_text = temp_text.strip()
                        if temp_text != "":
                            extracted_text = extracted_text + temp_text + ' '

                    p_count = p_count + 1

                if not y_flag:                              # Ja nav latgaļu valoda, pievieno tekstiem.
                    all_text.append([id, extracted_text])
                file_num = file_num + 1

Pēc teksta iegūšanas un sakārtošanas notiek tā vārdu marķēšana, kā arī biežumu aprēķināšana.

In [ ]:
import stanza
import json

pipeline = stanza.Pipeline(lang='lv', processors='tokenize,pos,lemma,depparse')

result_list = []
for i in range(len(all_text)):
    tagged_text = pipeline(all_text[i][1])
    freq_count = tagged_text.num_words
    freq_list = {}
    subject_list = {}        # Uzglabā tieši teikuma priekšmetu biežumu.
    for sent in tagged_text.sentences:
        for word in sent.words:
            if word.upos == "NOUN" or word.upos == "PROPN":
                if word.deprel == "nsubj":
                    if word.lemma in subject_list:
                        subject_list[word.lemma] = subject_list[word.lemma] + 1
                    else:
                        subject_list[word.lemma] = 0
                
                if word.lemma in freq_list:
                    freq_list[word.lemma] = freq_list[word.lemma] + 1
                else:
                    freq_list[word.lemma] = 0

    word_list = set()
    target_count = 2         # Īsākos tekstos varonis parādīsies mazāk reižu, bet garākos tekstos varoņi parādās vairākkārt.
    if freq_count > 2000:
        target_count = 5
    elif freq_count > 1000:
        target_count = 4
    elif freq_count > 400:
        target_count = 3

    for word in subject_list:
        if (freq_list[word] >= target_count) and (subject_list[word] / freq_list[word] > 1/3):      # Var būt arī citi svarīgi objekti, bet teikuma priekšmets parasti būs darītājs.
            word_list.add(word)
    
    result_object = { "text": all_text[i][1], "characters": list(word_list) }
    result_list.append(result_object)

with open("results.json", 'w') as file:
    json.dump(result_list, file, indent=4)


Pēc rezultātu iegūšanas konkrētie rezultāti ir jānovērtē. Testa datiem pareizie varoņu saraksti bija jāsaliek pašai, tāpēc testa datu apjoms ir salīdzinoši neliels - tikai 110 testa piemēri. No šīm pasakām tika paņemti daži visbiežāk atrodamie varoņi un tiem tika aprēķināts rīka akurātums. Tika paņemti arī īpašvārdi, jo tiem var izrēķināt arī precizitāti un pārklājumu.

In [ ]:
# Test block.
print(len(result_list))

with open('test.json', 'r') as file:
    test_list = json.load(file)
    print(len(test_list))


2850
2850
110
